
# BayesMMRL mechanism diagnosis — fixed5

这个 notebook 的关键修复是：

- `case_root` 仍然使用真实权重路径，例如 `.../ViT-B-16/...`
- `cfg.MODEL.BACKBONE.NAME` 必须使用 CLIP registry 真实名字，例如 `ViT-B/16`
- 不再硬编码从 `output_refactor` 解析路径；同时支持 `output_refactor` 和 `output_sweeps`
- 优先使用 `CASES` 里显式写的真实 cfg 字段，避免从目录名反推配置造成错误
- 先单独测试 build，再跑 audit，避免错误被后续机制分析输出淹没

运行顺序：

1. 运行 Cell 1 初始化路径和 `CASES`
2. 运行 Cell 2 导入环境、注册 trainer/dataset、定义 `setup_cfg`
3. 运行 Cell 3 定义 metadata、args、build、audit
4. 运行 Cell 4 只测试 build
5. build 成功后运行 Cell 5 audit


In [1]:

from pathlib import Path

# ============================================================
# Cell 1: 初始化路径、CASES、分析参数
# ============================================================

# ======== 修改这里 ========
REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATA_ROOT = REPO_ROOT / "DATASETS"
DASSL_ROOT = Path("/root/autodl-tmp/Dassl.pytorch").expanduser().resolve()

CASES = [
    {
        "name": "MMRL_caltech101_16shot_seed1",
        "case_root": REPO_ROOT / "output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1",

        # 这些是真实 cfg 值，不是路径名
        "trainer": "BayesMMRL",
        "dataset": "caltech101",
        "shots": 16,
        "backbone": "ViT-B/16",
        "seed": 1,
        "config": "rep_zero_sig-diagonal_pstd-1.0_kl-5e-2",
    },
    {
        "name": "BayesMMRL_caltech101_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1",

        "trainer": "BayesMMRL",
        "dataset": "caltech101",
        "shots": 16,
        "backbone": "ViT-B/16",
        "seed": 1,
        "config": "default",
    },
]

SPLIT = "test"          # "test" or "val"
MAX_BATCHES = None      # debug 时可设为 2 / 5
BAYES_N_MC = 20         # 分析用 MC samples 数，不影响训练
DEVICE = None           # None 表示自动用 trainer.device

SAVE_DIR = REPO_ROOT / "analysis_outputs" / "bayes_mmrl_mechanism_v2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.exists(), f"REPO_ROOT 不存在: {REPO_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT 不存在: {DATA_ROOT}"
assert DASSL_ROOT.exists(), f"DASSL_ROOT 不存在: {DASSL_ROOT}"

for case in CASES:
    assert Path(case["case_root"]).exists(), f"case_root 不存在: {case['case_root']}"

print("REPO_ROOT =", REPO_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("DASSL_ROOT=", DASSL_ROOT)
print("SAVE_DIR  =", SAVE_DIR)
print("CASES     =", len(CASES))

for case in CASES:
    print("\nCASE:", case["name"])
    print("  case_root =", case["case_root"])
    print("  trainer   =", case["trainer"])
    print("  dataset   =", case["dataset"])
    print("  shots     =", case["shots"])
    print("  backbone  =", case["backbone"])
    print("  config    =", case["config"])
    print("  seed      =", case["seed"])


REPO_ROOT = /root/autodl-tmp/MMRL
DATA_ROOT = /root/autodl-tmp/MMRL/DATASETS
DASSL_ROOT= /root/autodl-tmp/Dassl.pytorch
SAVE_DIR  = /root/autodl-tmp/MMRL/analysis_outputs/bayes_mmrl_mechanism_v2
CASES     = 2

CASE: MMRL_caltech101_16shot_seed1
  case_root = /root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1
  trainer   = BayesMMRL
  dataset   = caltech101
  shots     = 16
  backbone  = ViT-B/16
  config    = rep_zero_sig-diagonal_pstd-1.0_kl-5e-2
  seed      = 1

CASE: BayesMMRL_caltech101_16shot_seed1
  case_root = /root/autodl-tmp/MMRL/output_refactor/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1
  trainer   = BayesMMRL
  dataset   = caltech101
  shots     = 16
  backbone  = ViT-B/16
  config    = default
  seed      = 1


In [2]:

# ============================================================
# Cell 2: imports、注册 trainer/dataset、setup_cfg
# ============================================================

import os
import sys
import json
import random
import warnings
import importlib
import pkgutil
from pathlib import Path
from types import SimpleNamespace

import torch
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. 加入项目路径
# ------------------------------------------------------------

for p in [REPO_ROOT, DASSL_ROOT]:
    p = str(Path(p).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path inserted:")
print("  ", REPO_ROOT)
print("  ", DASSL_ROOT)


# ------------------------------------------------------------
# 2. 导入 Dassl 基础组件
# ------------------------------------------------------------

from dassl.config import get_cfg_default
from dassl.engine import build_trainer
from dassl.engine import TRAINER_REGISTRY


# ------------------------------------------------------------
# 3. 导入项目里的 extend_cfg
#    BayesMMRL 这类 trainer 往往需要自定义 cfg 字段。
# ------------------------------------------------------------

_EXTEND_CFG_FOUND = False

try:
    from train import extend_cfg
    _EXTEND_CFG_FOUND = True
    print("Loaded extend_cfg from train.py")
except Exception as e:
    print("Could not import extend_cfg from train.py:", repr(e))

if not _EXTEND_CFG_FOUND:
    try:
        from tools.train import extend_cfg
        _EXTEND_CFG_FOUND = True
        print("Loaded extend_cfg from tools/train.py")
    except Exception as e:
        print("Could not import extend_cfg from tools/train.py:", repr(e))

if not _EXTEND_CFG_FOUND:
    warnings.warn(
        "没有找到项目里的 extend_cfg。"
        "如果 BayesMMRL 需要自定义 cfg 字段，后面 setup_cfg 可能会报错。"
    )

    def extend_cfg(cfg):
        return cfg


# ------------------------------------------------------------
# 4. 自动导入 datasets / trainers 包，确保 registry 注册完成
# ------------------------------------------------------------

def import_all_submodules(package_name, max_depth=6):
    """
    递归导入 package 下子模块。
    目的：触发 DATASET_REGISTRY / TRAINER_REGISTRY 注册。

    有些模块可能因为可选依赖失败；这里仅 warning，不中断。
    """
    try:
        package = importlib.import_module(package_name)
    except Exception as e:
        print(f"[WARN] Cannot import package {package_name}: {repr(e)}")
        return

    if not hasattr(package, "__path__"):
        return

    prefix = package.__name__ + "."
    for module_info in pkgutil.walk_packages(package.__path__, prefix=prefix):
        full_name = module_info.name
        # 控制递归深度，避免误导入过深工具模块
        depth = full_name.count(".") - package_name.count(".")
        if depth > max_depth:
            continue
        try:
            importlib.import_module(full_name)
        except Exception as e:
            print(f"[WARN] Cannot import {full_name}: {repr(e)}")


import_all_submodules("datasets")
import_all_submodules("trainers")

try:
    registered_trainers = list(TRAINER_REGISTRY._obj_map.keys())
except Exception:
    registered_trainers = []

print("\nRegistered trainers:")
for t in registered_trainers:
    print("  ", t)


# ------------------------------------------------------------
# 5. cfg 工具函数
# ------------------------------------------------------------

def reset_cfg(cfg, args):
    """
    按 Dassl 官方 train.py 的习惯，从 args 覆盖 cfg。
    """
    if getattr(args, "root", None):
        cfg.DATASET.ROOT = args.root

    if getattr(args, "output_dir", None):
        cfg.OUTPUT_DIR = args.output_dir

    if getattr(args, "resume", None):
        cfg.RESUME = args.resume

    if getattr(args, "seed", None) is not None:
        cfg.SEED = int(args.seed)

    if getattr(args, "source_domains", None):
        cfg.DATASET.SOURCE_DOMAINS = args.source_domains

    if getattr(args, "target_domains", None):
        cfg.DATASET.TARGET_DOMAINS = args.target_domains

    if getattr(args, "transforms", None):
        cfg.INPUT.TRANSFORMS = args.transforms

    if getattr(args, "trainer", None):
        cfg.TRAINER.NAME = args.trainer

    if getattr(args, "backbone", None):
        cfg.MODEL.BACKBONE.NAME = args.backbone

    if getattr(args, "head", None):
        cfg.MODEL.HEAD.NAME = args.head


def setup_cfg(args):
    """
    完整 setup_cfg。

    merge 顺序：
    1. get_cfg_default()
    2. extend_cfg(cfg)
    3. dataset_config_file
    4. config_file 或 saved_config_file
    5. reset_cfg(args)
    6. merge_from_list(args.opts)
    7. freeze
    """
    cfg = get_cfg_default()
    extend_cfg(cfg)

    dataset_config_file = getattr(args, "dataset_config_file", "")
    config_file = getattr(args, "config_file", "")

    if dataset_config_file:
        dataset_config_file = str(dataset_config_file)
        if Path(dataset_config_file).exists():
            print("Merging dataset config:", dataset_config_file)
            cfg.merge_from_file(dataset_config_file)
        else:
            print("[WARN] dataset_config_file 不存在，跳过:", dataset_config_file)

    if config_file:
        config_file = str(config_file)
        if Path(config_file).exists():
            print("Merging trainer/saved config:", config_file)
            cfg.merge_from_file(config_file)
        else:
            print("[WARN] config_file 不存在，跳过:", config_file)

    reset_cfg(cfg, args)

    opts = getattr(args, "opts", [])
    if opts:
        if len(opts) % 2 != 0:
            raise ValueError(f"args.opts 长度必须是偶数，当前为 {len(opts)}: {opts}")

        print("Merging opts:")
        for k, v in zip(opts[0::2], opts[1::2]):
            print("  ", k, "=", v)
        cfg.merge_from_list(opts)

    cfg.freeze()
    return cfg


sys.path inserted:
   /root/autodl-tmp/MMRL
   /root/autodl-tmp/Dassl.pytorch
Could not import extend_cfg from train.py: ModuleNotFoundError("No module named 'train'")
Loaded extend_cfg from tools/train.py

Registered trainers:
   SE
   MCD
   MME
   ADDA
   CDAC
   DAEL
   DANN
   AdaBN
   M3SDA
   SourceOnly
   DDAIG
   DAELDG
   Vanilla
   CrossGrad
   DomainMix
   EntMin
   FixMatch
   MixMatch
   MeanTeacher
   SupBaseline
   RefactorRunner


In [3]:

# ============================================================
# Cell 3: metadata、args、build、audit
# ============================================================

from pathlib import Path
from types import SimpleNamespace


# ============================================================
# 1. 路径名和 cfg 名的转换
# ============================================================

def decode_backbone_name(x):
    """
    把路径里的安全文件夹名转换成 CLIP / Dassl 真正需要的 backbone 名称。

    路径里:
        ViT-B-16

    cfg 里:
        ViT-B/16
    """
    mapping = {
        "ViT-B-16": "ViT-B/16",
        "ViT-B-32": "ViT-B/32",
        "ViT-L-14": "ViT-L/14",
        "ViT-L-14-336px": "ViT-L/14@336px",
    }
    return mapping.get(str(x), str(x))


def encode_backbone_folder(x):
    """
    把真实 backbone 名转换成路径文件夹名。
    """
    mapping = {
        "ViT-B/16": "ViT-B-16",
        "ViT-B/32": "ViT-B-32",
        "ViT-L/14": "ViT-L-14",
        "ViT-L/14@336px": "ViT-L-14-336px",
    }
    return mapping.get(str(x), str(x))


def canonical_dataset_name(x):
    """
    路径中的 dataset 名通常是小写；
    Dassl registry 里的 DATASET.NAME 通常是 CamelCase。
    """
    mapping = {
        "caltech101": "Caltech101",
        "oxford_pets": "OxfordPets",
        "stanford_cars": "StanfordCars",
        "oxford_flowers": "OxfordFlowers",
        "food101": "Food101",
        "fgvc_aircraft": "FGVCAircraft",
        "sun397": "SUN397",
        "dtd": "DescribableTextures",
        "eurosat": "EuroSAT",
        "ucf101": "UCF101",
        "imagenet": "ImageNet",
    }
    return mapping.get(str(x), str(x))


def backbone_to_config_stem(backbone):
    """
    一些 trainer config 文件可能叫 vit_b16.yaml / vit_b32.yaml。
    """
    mapping = {
        "ViT-B/16": "vit_b16",
        "ViT-B/32": "vit_b32",
        "ViT-L/14": "vit_l14",
        "ViT-L/14@336px": "vit_l14_336px",
    }
    return mapping.get(str(backbone), encode_backbone_folder(backbone))


# ============================================================
# 2. 自动找 saved config / dataset config / trainer config
# ============================================================

def find_existing_file(candidates):
    for p in candidates:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return str(p)
    return ""


def find_saved_config_file(case_root):
    """
    优先找训练输出目录里保存的 cfg。
    如果存在，这比从路径反推配置可靠。
    """
    case_root = Path(case_root)
    candidates = [
        case_root / "config.yaml",
        case_root / "config.yml",
        case_root / "cfg.yaml",
        case_root / "cfg.yml",
        case_root / "args.yaml",
        case_root / "args.yml",
    ]
    return find_existing_file(candidates)


def find_dataset_config_file(dataset):
    """
    尝试寻找 configs/datasets/caltech101.yaml 这类文件。
    找不到也不致命，因为后面会用 opts 设置 DATASET.NAME。
    """
    dataset = str(dataset)
    canonical = canonical_dataset_name(dataset)
    candidates = [
        REPO_ROOT / "configs" / "datasets" / f"{dataset}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{dataset.lower()}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{canonical}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{canonical.lower()}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{dataset}.yml",
        REPO_ROOT / "configs" / "datasets" / f"{dataset.lower()}.yml",
    ]
    return find_existing_file(candidates)


def find_trainer_config_file(meta):
    """
    尝试寻找 trainer config。

    注意：
    output_sweeps 里的 config 名可能只是实验标签，
    未必真的存在一个同名 yaml。
    所以找不到不直接报错。
    """
    trainer = meta["trainer"]
    config = str(meta.get("config", ""))
    backbone = meta["backbone"]
    backbone_folder = meta["backbone_folder"]
    backbone_stem = backbone_to_config_stem(backbone)

    candidates = [
        REPO_ROOT / "configs" / "trainers" / trainer / f"{config}.yaml",
        REPO_ROOT / "configs" / "trainers" / trainer / f"{backbone_stem}.yaml",
        REPO_ROOT / "configs" / "trainers" / trainer / f"{backbone_folder}.yaml",
        REPO_ROOT / "configs" / "trainers" / trainer / "default.yaml",

        REPO_ROOT / "configs" / trainer / f"{config}.yaml",
        REPO_ROOT / "configs" / trainer / f"{backbone_stem}.yaml",
        REPO_ROOT / "configs" / trainer / f"{backbone_folder}.yaml",
        REPO_ROOT / "configs" / trainer / "default.yaml",

        REPO_ROOT / "configs" / "trainers" / trainer / f"{config}.yml",
        REPO_ROOT / "configs" / "trainers" / trainer / f"{backbone_stem}.yml",
        REPO_ROOT / "configs" / "trainers" / trainer / f"{backbone_folder}.yml",
        REPO_ROOT / "configs" / "trainers" / trainer / "default.yml",
    ]

    return find_existing_file(candidates)


# ============================================================
# 3. 从 case_root 解析 metadata
# ============================================================

def infer_case_metadata(case_root, case=None):
    """
    支持：

    output_refactor/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1

    和：

    output_sweeps/.../confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/xxx/seed1

    关键：
    - backbone_folder 是路径名：ViT-B-16
    - backbone 是 cfg 名：ViT-B/16
    """
    case = case or {}
    case_root = Path(case_root).expanduser().resolve()
    parts = case_root.parts

    anchors = [
        i for i, p in enumerate(parts)
        if p == "FS" and i + 1 < len(parts) and parts[i + 1] == "fewshot_train"
    ]

    if not anchors:
        raise ValueError(
            "无法从路径中找到 FS/fewshot_train 结构。\n"
            f"case_root = {case_root}"
        )

    i = anchors[-1]

    try:
        method = parts[i - 1]
        protocol = parts[i]
        train_mode = parts[i + 1]
        dataset = parts[i + 2]
        shots_str = parts[i + 3]
        backbone_folder = parts[i + 4]
        config = parts[i + 5]
        seed_str = parts[i + 6]
    except IndexError as e:
        raise ValueError(
            "路径结构不完整，无法解析 method/dataset/shots/backbone/config/seed。\n"
            f"case_root = {case_root}"
        ) from e

    if not shots_str.startswith("shots_"):
        raise ValueError(f"shots 字段异常: {shots_str}, case_root={case_root}")

    if not seed_str.startswith("seed"):
        raise ValueError(f"seed 字段异常: {seed_str}, case_root={case_root}")

    shots = int(shots_str.replace("shots_", ""))
    seed = int(seed_str.replace("seed", ""))

    if "output_refactor" in parts:
        output_family = "output_refactor"
    elif "output_sweeps" in parts:
        output_family = "output_sweeps"
    else:
        output_family = "unknown"

    meta = {
        "name": case.get("name", case_root.name),
        "case_root": case_root,

        "output_family": output_family,
        "method": method,
        "trainer": method,
        "protocol": protocol,
        "train_mode": train_mode,

        "dataset": dataset,
        "dataset_name": canonical_dataset_name(dataset),

        "shots": shots,

        "backbone_folder": backbone_folder,
        "backbone": decode_backbone_name(backbone_folder),

        "config": config,
        "seed": seed,
    }

    # CASES 里显式字段优先级最高
    for key in ["trainer", "method", "dataset", "shots", "backbone", "config", "seed"]:
        if key in case and case[key] is not None:
            meta[key] = case[key]

    meta["shots"] = int(meta["shots"])
    meta["seed"] = int(meta["seed"])
    meta["backbone"] = decode_backbone_name(meta["backbone"])
    meta["backbone_folder"] = encode_backbone_folder(meta["backbone"])
    meta["dataset_name"] = canonical_dataset_name(meta["dataset"])

    saved_config = find_saved_config_file(case_root)
    dataset_config = find_dataset_config_file(meta["dataset"])
    trainer_config = find_trainer_config_file(meta)

    # 优先使用训练输出目录保存的完整 config；
    # 如果没有，再尝试 trainer config。
    meta["saved_config_file"] = saved_config
    meta["dataset_config_file"] = dataset_config
    meta["trainer_config_file"] = trainer_config
    meta["config_file"] = saved_config or trainer_config

    return meta


# ============================================================
# 4. opts 设置工具
# ============================================================

def _set_opt(opts, key, value):
    """
    在 opts 里设置 key value。
    如果 key 已存在，替换它后面的 value；
    如果不存在，追加。
    """
    opts = [] if opts is None else list(opts)

    if key in opts:
        idx = opts.index(key)
        if idx + 1 >= len(opts):
            raise ValueError(f"opts 中 {key} 后面没有 value")
        opts[idx + 1] = str(value)
    else:
        opts.extend([key, str(value)])

    return opts


# ============================================================
# 5. 完整 make_args_from_case
# ============================================================

def make_args_from_case(meta):
    """
    构造一个类似 argparse.parse_args() 返回的对象。

    重点：
    - case_root 只给 load_model 和 output_dir 用
    - cfg.MODEL.BACKBONE.NAME 必须用 meta["backbone"]，也就是 ViT-B/16
    - 不允许把 backbone_folder 的 ViT-B-16 塞进 cfg
    """
    case_root = Path(meta["case_root"]).expanduser().resolve()

    args = SimpleNamespace()

    # Dassl 常见参数
    args.root = str(DATA_ROOT)
    args.output_dir = str(case_root)
    args.resume = ""
    args.seed = int(meta["seed"])

    args.source_domains = None
    args.target_domains = None
    args.transforms = None

    args.trainer = meta["trainer"]
    args.backbone = meta["backbone"]
    args.head = ""

    args.dataset_config_file = meta.get("dataset_config_file", "")
    args.config_file = meta.get("config_file", "")

    args.eval_only = True
    args.model_dir = str(case_root)
    args.load_epoch = None
    args.no_train = True

    # opts 是最终兜底，优先级最高
    opts = []

    opts = _set_opt(opts, "TRAINER.NAME", meta["trainer"])
    opts = _set_opt(opts, "DATASET.ROOT", str(DATA_ROOT))
    opts = _set_opt(opts, "DATASET.NAME", meta["dataset_name"])
    opts = _set_opt(opts, "DATASET.NUM_SHOTS", int(meta["shots"]))
    opts = _set_opt(opts, "MODEL.BACKBONE.NAME", meta["backbone"])
    opts = _set_opt(opts, "SEED", int(meta["seed"]))
    opts = _set_opt(opts, "OUTPUT_DIR", str(case_root))

    args.opts = opts

    return args


# ============================================================
# 6. build 前 debug 检查
# ============================================================

def debug_print_meta(meta):
    print("\n[DEBUG meta]")
    print("name                =", meta["name"])
    print("case_root           =", meta["case_root"])
    print("output_family       =", meta.get("output_family"))
    print("trainer             =", meta["trainer"])
    print("dataset(path)       =", meta["dataset"])
    print("dataset(cfg)        =", meta["dataset_name"])
    print("shots               =", meta["shots"])
    print("backbone_folder     =", meta["backbone_folder"])
    print("backbone(cfg)       =", meta["backbone"])
    print("config              =", meta["config"])
    print("seed                =", meta["seed"])
    print("saved_config_file   =", meta.get("saved_config_file") or "<none>")
    print("dataset_config_file =", meta.get("dataset_config_file") or "<none>")
    print("trainer_config_file =", meta.get("trainer_config_file") or "<none>")
    print("config_file used    =", meta.get("config_file") or "<none>")


def debug_print_args(args):
    print("\n[DEBUG args]")
    print("args.root                =", args.root)
    print("args.output_dir          =", args.output_dir)
    print("args.trainer             =", args.trainer)
    print("args.backbone            =", args.backbone)
    print("args.seed                =", args.seed)
    print("args.dataset_config_file =", args.dataset_config_file or "<none>")
    print("args.config_file         =", args.config_file or "<none>")
    print("args.opts:")
    for k, v in zip(args.opts[0::2], args.opts[1::2]):
        print("  ", k, "=", v)


def debug_print_cfg(cfg, meta):
    print("\n[DEBUG cfg]")
    print("cfg.TRAINER.NAME         =", cfg.TRAINER.NAME)
    print("cfg.DATASET.NAME         =", cfg.DATASET.NAME)
    print("cfg.DATASET.ROOT         =", cfg.DATASET.ROOT)
    print("cfg.MODEL.BACKBONE.NAME  =", cfg.MODEL.BACKBONE.NAME)
    print("cfg.SEED                 =", cfg.SEED)
    print("cfg.OUTPUT_DIR           =", cfg.OUTPUT_DIR)

    if cfg.MODEL.BACKBONE.NAME != meta["backbone"]:
        raise AssertionError(
            "backbone 没有真正进入 cfg。\n"
            f"cfg.MODEL.BACKBONE.NAME = {cfg.MODEL.BACKBONE.NAME}\n"
            f"meta['backbone']        = {meta['backbone']}\n"
        )

    if cfg.MODEL.BACKBONE.NAME == meta["backbone_folder"]:
        raise AssertionError(
            "cfg.MODEL.BACKBONE.NAME 仍然是路径文件夹名，这是错的。\n"
            f"当前 cfg.MODEL.BACKBONE.NAME = {cfg.MODEL.BACKBONE.NAME}\n"
            f"应该是 {meta['backbone']}"
        )

    if "ViT" in cfg.MODEL.BACKBONE.NAME:
        assert "/" in cfg.MODEL.BACKBONE.NAME, (
            "ViT backbone 必须包含 /，例如 ViT-B/16，"
            f"当前是 {cfg.MODEL.BACKBONE.NAME}"
        )


# ============================================================
# 7. 完整 build_trainer_for_case
# ============================================================

def build_trainer_for_case(case):
    case_root = Path(case["case_root"]).expanduser().resolve()

    if not case_root.exists():
        raise FileNotFoundError(f"case_root 不存在: {case_root}")

    meta = infer_case_metadata(case_root, case)

    debug_print_meta(meta)

    args = make_args_from_case(meta)

    debug_print_args(args)

    cfg = setup_cfg(args)

    debug_print_cfg(cfg, meta)

    trainer = build_trainer(cfg)

    # 注意：这里必须用原始权重目录。
    # 不要把路径中的 ViT-B-16 改成 ViT-B/16。
    print("\nLoading model from:", case_root)
    trainer.load_model(str(case_root))
    trainer.set_model_mode("eval")

    return trainer, meta


# ============================================================
# 8. audit 函数
# ============================================================

def _tensor_stats(x):
    if not torch.is_tensor(x):
        return {}
    x = x.detach().float().cpu()
    if x.numel() == 0:
        return {
            "shape": tuple(x.shape),
            "numel": 0,
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
        }
    return {
        "shape": tuple(x.shape),
        "numel": int(x.numel()),
        "mean": float(x.mean().item()),
        "std": float(x.std(unbiased=False).item()),
        "min": float(x.min().item()),
        "max": float(x.max().item()),
    }


def find_checkpoint_files(case_root):
    case_root = Path(case_root)
    patterns = [
        "*.pth",
        "*.pth.tar",
        "**/*.pth",
        "**/*.pth.tar",
        "*.pt",
        "**/*.pt",
    ]
    files = []
    for pat in patterns:
        files.extend(case_root.glob(pat))
    files = sorted(set(files), key=lambda p: str(p))
    return files


def summarize_checkpoint_keys(case_root, max_files=20):
    rows = []
    ckpt_files = find_checkpoint_files(case_root)

    for p in ckpt_files[:max_files]:
        row = {
            "file": str(p),
            "size_mb": round(p.stat().st_size / 1024 / 1024, 3),
            "load_ok": False,
            "top_keys": "",
            "state_dict_keys": 0,
            "state_dict_prefixes": "",
            "error": "",
        }

        try:
            obj = torch.load(str(p), map_location="cpu")
            row["load_ok"] = True

            if isinstance(obj, dict):
                top_keys = list(obj.keys())
                row["top_keys"] = ", ".join(map(str, top_keys[:30]))

                sd = None
                for key in ["state_dict", "model", "model_state_dict", "net"]:
                    if key in obj and isinstance(obj[key], dict):
                        sd = obj[key]
                        break
                if sd is None:
                    tensor_dict = {k: v for k, v in obj.items() if torch.is_tensor(v)}
                    if tensor_dict:
                        sd = tensor_dict

                if isinstance(sd, dict):
                    keys = list(sd.keys())
                    row["state_dict_keys"] = len(keys)
                    prefixes = {}
                    for k in keys:
                        pref = str(k).split(".")[0]
                        prefixes[pref] = prefixes.get(pref, 0) + 1
                    row["state_dict_prefixes"] = ", ".join(
                        f"{k}:{v}" for k, v in sorted(prefixes.items(), key=lambda kv: -kv[1])[:20]
                    )
            else:
                row["top_keys"] = type(obj).__name__

        except Exception as e:
            row["error"] = repr(e)

        rows.append(row)

    if not rows:
        rows.append({
            "file": "<no checkpoint files found>",
            "size_mb": np.nan,
            "load_ok": False,
            "top_keys": "",
            "state_dict_keys": 0,
            "state_dict_prefixes": "",
            "error": "",
        })

    return pd.DataFrame(rows)


def summarize_model_params(trainer, max_rows=2000):
    model = getattr(trainer, "model", None)
    if model is None:
        return pd.DataFrame([{"name": "<trainer has no .model>", "error": ""}])

    rows = []

    for name, p in model.named_parameters():
        st = _tensor_stats(p)
        rows.append({
            "kind": "param",
            "name": name,
            "requires_grad": bool(p.requires_grad),
            **st,
        })

    for name, b in model.named_buffers():
        st = _tensor_stats(b)
        rows.append({
            "kind": "buffer",
            "name": name,
            "requires_grad": False,
            **st,
        })

    df = pd.DataFrame(rows)
    if len(df) > max_rows:
        return df.head(max_rows).copy()
    return df


def summarize_bayes_posterior_like(trainer):
    """
    抽取名字里像 Bayesian/posterior 参数的 tensor。
    这不是强假设，只做诊断。
    """
    df = summarize_model_params(trainer)
    if "name" not in df.columns:
        return df

    keywords = [
        "mu", "rho", "sigma", "std", "logvar", "log_var",
        "posterior", "prior", "kl", "bayes", "alpha", "beta"
    ]

    mask = df["name"].astype(str).str.lower().apply(
        lambda s: any(k in s for k in keywords)
    )

    out = df[mask].copy()
    if len(out) == 0:
        return pd.DataFrame([{
            "note": "没有在 model.named_parameters()/buffers() 中找到名字像 posterior/Bayesian 的 tensor",
        }])

    return out


def audit_case(trainer, meta):
    cfg = trainer.cfg

    cfg_rows = [
        {"key": "name", "value": meta.get("name")},
        {"key": "case_root", "value": str(meta.get("case_root"))},
        {"key": "output_family", "value": meta.get("output_family")},
        {"key": "trainer(meta)", "value": meta.get("trainer")},
        {"key": "dataset(path)", "value": meta.get("dataset")},
        {"key": "dataset(cfg meta)", "value": meta.get("dataset_name")},
        {"key": "shots", "value": meta.get("shots")},
        {"key": "backbone_folder(path)", "value": meta.get("backbone_folder")},
        {"key": "backbone(meta cfg)", "value": meta.get("backbone")},
        {"key": "config", "value": meta.get("config")},
        {"key": "seed", "value": meta.get("seed")},
        {"key": "saved_config_file", "value": meta.get("saved_config_file") or "<none>"},
        {"key": "dataset_config_file", "value": meta.get("dataset_config_file") or "<none>"},
        {"key": "trainer_config_file", "value": meta.get("trainer_config_file") or "<none>"},
        {"key": "config_file used", "value": meta.get("config_file") or "<none>"},
        {"key": "cfg.TRAINER.NAME", "value": cfg.TRAINER.NAME},
        {"key": "cfg.DATASET.NAME", "value": cfg.DATASET.NAME},
        {"key": "cfg.DATASET.ROOT", "value": cfg.DATASET.ROOT},
        {"key": "cfg.MODEL.BACKBONE.NAME", "value": cfg.MODEL.BACKBONE.NAME},
        {"key": "cfg.SEED", "value": cfg.SEED},
        {"key": "cfg.OUTPUT_DIR", "value": cfg.OUTPUT_DIR},
    ]

    cfg_df = pd.DataFrame(cfg_rows)
    keys_df = summarize_checkpoint_keys(meta["case_root"])
    post_df = summarize_bayes_posterior_like(trainer)

    return cfg_df, keys_df, post_df


In [4]:

# ============================================================
# Cell 4: 先只测试 build，不跑 audit
# ============================================================

trainers = {}

for case in CASES:
    print("\n" + "=" * 100)
    print(f"=== Building {case['name']} ===")
    print("=" * 100)

    trainer, meta = build_trainer_for_case(case)
    trainers[meta["name"]] = (trainer, meta)

print("\nBuilt trainers:")
for name in trainers:
    print("  ", name)



=== Building MMRL_caltech101_16shot_seed1 ===

[DEBUG meta]
name                = MMRL_caltech101_16shot_seed1
case_root           = /root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1
output_family       = output_sweeps
trainer             = BayesMMRL
dataset(path)       = caltech101
dataset(cfg)        = Caltech101
shots               = 16
backbone_folder     = ViT-B-16
backbone(cfg)       = ViT-B/16
config              = rep_zero_sig-diagonal_pstd-1.0_kl-5e-2
seed                = 1
saved_config_file   = <none>
dataset_config_file = /root/autodl-tmp/MMRL/configs/datasets/caltech101.yaml
trainer_config_file = <none>
config_file used    = <none>

[DEBUG args]
args.root                = /root/autodl-tmp/MMRL/DATASETS
args.output_dir          = /root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech10

ValueError: The requested one is expected to belong to ['SE', 'MCD', 'MME', 'ADDA', 'CDAC', 'DAEL', 'DANN', 'AdaBN', 'M3SDA', 'SourceOnly', 'DDAIG', 'DAELDG', 'Vanilla', 'CrossGrad', 'DomainMix', 'EntMin', 'FixMatch', 'MixMatch', 'MeanTeacher', 'SupBaseline', 'RefactorRunner'], but got [BayesMMRL] (do you mean [MME]?)

In [5]:
from pathlib import Path

for case in CASES:
    root = Path(case["case_root"])
    print("\n" + "=" * 100)
    print(case["name"])
    print("case_root =", root)

    candidates = [
        root / "config.yaml",
        root / "config.yml",
        root / "cfg.yaml",
        root / "cfg.yml",
        root / "args.yaml",
        root / "args.yml",
        root / "log.txt",
        root / "stdout.txt",
    ]

    for p in candidates:
        if p.exists():
            print("\nFOUND:", p)
            try:
                txt = p.read_text(errors="ignore")
                for line in txt.splitlines():
                    if (
                        "TRAINER" in line
                        or "Trainer" in line
                        or "trainer" in line
                        or "BayesMMRL" in line
                        or "RefactorRunner" in line
                        or "BACKBONE" in line
                        or "ViT-B" in line
                    ):
                        print(line)
            except Exception as e:
                print("read failed:", repr(e))


MMRL_caltech101_16shot_seed1
case_root = /root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1

FOUND: /root/autodl-tmp/MMRL/output_sweeps/bayes_mmrl_rep_tokens/coarse3/confirm_stage/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/rep_zero_sig-diagonal_pstd-1.0_kl-5e-2/seed1/log.txt
method: BayesMMRL
opts: ['DATASET.NUM_SHOTS', '16', 'DATASET.SUBSAMPLE_CLASSES', 'all', 'MODEL.BACKBONE.NAME', 'ViT-B/16', 'BAYES_MMRL.ALPHA', '0.7', 'BAYES_MMRL.REG_WEIGHT', '0.5', 'BAYES_MMRL.N_REP_TOKENS', '5', 'BAYES_MMRL.REP_LAYERS', '[6,7,8,9,10,11,12]', 'BAYES_MMRL.REP_DIM', '512', 'BAYES_MMRL.N_MC_TRAIN', '3', 'BAYES_MMRL.N_MC_TEST', '10', 'BAYES_MMRL.EVAL_MODE', 'mc_predictive', 'BAYES_MMRL.EVAL_USE_POSTERIOR_MEAN', 'False', 'BAYES_MMRL.EVAL_AGGREGATION', 'logit_mean', 'BAYES_MMRL.KL_WARMUP_EPOCHS', '6', 'BAYES_MMRL.BAYES_TARGET', 'rep_tokens', 'BAYES_MMRL.REP_PRIO

In [ ]:

# ============================================================
# Cell 5: build 成功后再跑 audit
# ============================================================

audit_results = {}

for name, (trainer, meta) in trainers.items():
    print("\n" + "=" * 100)
    print(f"=== Auditing {name} ===")
    print("=" * 100)

    cfg_df, keys_df, post_df = audit_case(trainer, meta)

    audit_results[name] = {
        "cfg_df": cfg_df,
        "keys_df": keys_df,
        "post_df": post_df,
    }

    print("\n[AUDIT cfg]")
    display(cfg_df)

    print("\n[AUDIT checkpoint keys]")
    display(keys_df)

    print("\n[AUDIT posterior-like tensors]")
    display(post_df)


In [ ]:

# ============================================================
# Cell 6: 保存 audit 表到 SAVE_DIR
# ============================================================

for name, result in audit_results.items():
    safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in name)

    for table_name, df in result.items():
        out_csv = SAVE_DIR / f"{safe_name}_{table_name}.csv"
        df.to_csv(out_csv, index=False)
        print("saved:", out_csv)



## 如果仍然报错，优先看这三行

在 Cell 4 的输出里，必须看到：

```text
backbone_folder     = ViT-B-16
backbone(cfg)       = ViT-B/16
cfg.MODEL.BACKBONE.NAME  = ViT-B/16
```

如果 `cfg.MODEL.BACKBONE.NAME` 还是 `ViT-B-16`，说明没有重新运行 Cell 3，或者后面还有旧函数覆盖了新的 `build_trainer_for_case`。

如果错误变成 dataset/trainer registry 相关，说明下一步问题不是权重目录，而是：
- `trainers` 包没有成功 import，导致 `BayesMMRL` 没注册；
- 或者 `DATASET.NAME` 不匹配当前项目 registry；
- 或者项目的 `extend_cfg` 没有导入成功。
